## Use the generate_chat.py and try to pivot the chat messages

In [4]:
import pandas as pd
from generator.generate_chat import *

df = generate_chat_data_v1(num_to_generate=100)
df.head()

,room,username,message,timestamp
0,Room5,UserA,1. Tango6,2026-03-30 12:41:29.653442
1,Room5,UserA,2. TEST9,2026-03-30 12:41:29.663442
2,Room5,UserA,3. TEST9,2026-03-30 12:41:29.675442
3,Room5,UserA,4. None2,2026-03-30 12:41:29.689442
4,Room5,UserA,5. Expected Completion Time10: 2026-03-30T12:4...,2026-03-30 12:41:29.703442


### Simple naive merge

In [7]:
## That worked, so go ahead and write a function to do this
def pivot_multiline_messages(dataframe, prefixes, new_columns=None, order_by=["Room Name", "Timestamp"], message_column="Message"):
    if dataframe is None:
        raise ValueError("The input DataFrame is empty.")
    if prefixes is None or len(prefixes) == 0:
        raise ValueError("No prefixes provided.")

    ## If new columns weren't passed to us, go ahead and create some new prefixes
    if new_columns is None:
        new_columns = [f"msg_{i+1}" for i in range(len(prefixes))]

    to_return = {}
    tmp = dataframe[dataframe[message_column].str.startswith(prefixes[0])].copy().sort_values(by=order_by)
    for col in order_by:
        to_return[col] = tmp[col].values

    for i, prefix in enumerate(prefixes):
        tmp = dataframe[dataframe[message_column].str.startswith(prefix)].copy().sort_values(by=order_by)
        to_return[new_columns[i]] = tmp[message_column].str.slice(len(prefix)).values

    ## Return the our results
    return pd.DataFrame(to_return)

In [9]:
prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]

pivot_multiline_messages(df, prefixes, new_columns=new_columns, order_by=["room", "timestamp"], message_column="message")

,room,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5
0,Room1,2026-03-30 13:20:54.934442,Tango4,TEST2,NUMPY3,N/A5,ETA7: 2026-03-30T13:29:42.979442
1,Room1,2026-03-30 13:51:20.213442,Delta3,ESCORT6,TST10,None9,ETA7: 2026-03-30T13:53:56.227442
2,Room1,2026-03-30 14:10:03.332442,Tango8,TEST3,TEST9,None4,Time to Completion9: 2026-03-30T14:12:57.360442
3,Room1,2026-03-30 14:37:23.537442,Delta7,SMACK9,TST1,N/A3,ETA5: 2026-03-30T14:41:54.557442
4,Room1,2026-03-30 14:55:34.704442,Charlie1,SMACK8,NUMPY6,N/A6,Time to Completion5: 2026-03-30T15:00:10.728442
...,...,...,...,...,...,...,...
95,Room5,2026-03-30 20:08:31.924442,Delta7,ESCORT9,PANDAS3,N/A8,ETA1: 2026-03-30T20:10:22.952442
96,Room5,2026-03-30 20:10:22.962442,Bravo1,ESCORT10,TST9,None5,ETA10: 2026-03-30T20:14:37.985442
97,Room5,2026-03-30 20:14:38.000442,Charlie3,N/A8,TST7,N/A9,Expected Completion Time3: 2026-03-30T20:19:28...
98,Room5,2026-03-30 20:24:26.111442,Delta1,ESCORT10,NUMPY3,None10,TTC6: 2026-03-30T20:24:33.120442


### Simple naive merge v2

In [12]:
## That worked, so go ahead and write a function to do this
def pivot_multiline_messages(dataframe, prefixes, new_columns=None, order_by=["Room Name"], time_column="Timestamp", message_column="Message"):
    if dataframe is None:
        raise ValueError("The input DataFrame is empty.")
    if prefixes is None or len(prefixes) == 0:
        raise ValueError("No prefixes provided.")

    ## Add the timestamp column to the order by
    if order_by is None:
        order_by = [time_column]
    elif time_column in order_by:
        pass
    else:
        order_by.append(time_column)

    ## If new columns weren't passed to us, go ahead and create some new prefixes
    if new_columns is None:
        new_columns = [f"msg_{i+1}" for i in range(len(prefixes))]

    ## Create our to return
    to_return = {}

    ## Add the order by column(s) to our results,
    tmp = dataframe[dataframe[message_column].str.startswith(prefixes[0])].copy().sort_values(by=order_by)
    for col in order_by:
        to_return[col] = tmp[col].values
    ## Add the prefixes that we are looking for the the results
    for i, prefix in enumerate(prefixes):
        tmp = dataframe[dataframe[message_column].str.startswith(prefix)].copy().sort_values(by=order_by)
        to_return[new_columns[i]] = tmp[message_column].str.slice(len(prefix)).values

    ## Return the our results
    return pd.DataFrame(to_return)

In [13]:
prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]

pivot_multiline_messages(df, prefixes, new_columns=new_columns, order_by=["room", "timestamp"], time_column="timestamp", message_column="message")

,room,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5
0,Room1,2026-03-30 13:20:54.934442,Tango4,TEST2,NUMPY3,N/A5,ETA7: 2026-03-30T13:29:42.979442
1,Room1,2026-03-30 13:51:20.213442,Delta3,ESCORT6,TST10,None9,ETA7: 2026-03-30T13:53:56.227442
2,Room1,2026-03-30 14:10:03.332442,Tango8,TEST3,TEST9,None4,Time to Completion9: 2026-03-30T14:12:57.360442
3,Room1,2026-03-30 14:37:23.537442,Delta7,SMACK9,TST1,N/A3,ETA5: 2026-03-30T14:41:54.557442
4,Room1,2026-03-30 14:55:34.704442,Charlie1,SMACK8,NUMPY6,N/A6,Time to Completion5: 2026-03-30T15:00:10.728442
...,...,...,...,...,...,...,...
95,Room5,2026-03-30 20:08:31.924442,Delta7,ESCORT9,PANDAS3,N/A8,ETA1: 2026-03-30T20:10:22.952442
96,Room5,2026-03-30 20:10:22.962442,Bravo1,ESCORT10,TST9,None5,ETA10: 2026-03-30T20:14:37.985442
97,Room5,2026-03-30 20:14:38.000442,Charlie3,N/A8,TST7,N/A9,Expected Completion Time3: 2026-03-30T20:19:28...
98,Room5,2026-03-30 20:24:26.111442,Delta1,ESCORT10,NUMPY3,None10,TTC6: 2026-03-30T20:24:33.120442


### Try to perform a better merge, that isn't so naive about the timestamp and order

In [14]:

prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]

new_df = None
for i, prefix in enumerate(prefixes):
    tmpdf = df[df["message"].str.startswith(prefix)].copy()
    tmpdf[new_columns[i]] = tmpdf["message"].str.slice(len(prefix)).values
    tmpdf = tmpdf.drop(columns=["message"])

    if new_df is None:
        new_df = tmpdf
    else:
        new_df = pd.merge(new_df, tmpdf, on=["room", "timestamp", "username"], how="outer")

display(new_df)


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5
0,Room1,UserD,2026-03-30 13:20:54.934442,Tango4,NaN,NaN,NaN,NaN
1,Room1,UserD,2026-03-30 13:20:54.944442,NaN,TEST2,NaN,NaN,NaN
2,Room1,UserD,2026-03-30 13:20:54.955442,NaN,NaN,NUMPY3,NaN,NaN
3,Room1,UserD,2026-03-30 13:20:54.965442,NaN,NaN,NaN,N/A5,NaN
4,Room1,UserD,2026-03-30 13:20:54.979442,NaN,NaN,NaN,NaN,ETA7: 2026-03-30T13:29:42.979442
...,...,...,...,...,...,...,...,...
465,Room5,UserD,2026-03-30 20:41:01.196442,Echo6,NaN,NaN,NaN,NaN
466,Room5,UserD,2026-03-30 20:41:01.202442,NaN,TEST10,NaN,NaN,NaN
467,Room5,UserD,2026-03-30 20:41:01.203442,NaN,NaN,TST10,NaN,NaN
468,Room5,UserD,2026-03-30 20:41:01.208442,NaN,NaN,NaN,None9,NaN


### Write a function to try and fix the match, to be a little better

In [18]:
def convert_messages_to_ten_line(msg_df,
            prefixes=["1. ", "2. ", "3. ", "4. ", "5. ",
                      "6. ", "7. ", "8. ", "9. ", "10. "],
            new_columns=["msg_1", "msg_2", "msg_3", "msg_4", "msg_5",
                         "msg_6", "msg_7", "msg_8", "msg_9", "msg_10"],
            best_matches=False,
            msg_col="Message", timestmp_col="Timestamp",
            groupby_cnt_col="msg_cnt",
            match_group_col="match_group", time_diff_col="time_diff",
            groupby=["Room Name", "User Name"]):

    ## First go ahead and create the new columns and strip off our prefixes
    new_df = msg_df.copy()
    ## Loop through the prefixes and create the new columns
    for i, prefix in enumerate(prefixes):
        new_df[new_columns[i]] = new_df[new_df[msg_col].str.startswith(prefix)][msg_col].str.slice(len(prefix))

    ## Drop the message column, since we don't need it anymore
    new_df = new_df.drop(columns=[msg_col])

    ## Build our groupby (that includes the timestamp column)
    full_groupby = groupby.copy()
    full_groupby.append(timestmp_col)

    ## Add a column for our groupby count
    new_df[groupby_cnt_col] = 0

    ## GroupBy, first build a dic for our group by
    agg_dict = {}
    for col in new_df.columns:
        if not (col in full_groupby):
          agg_dict[col] = "first"
    agg_dict[groupby_cnt_col] = "count"
    new_df = new_df.groupby(full_groupby).agg(agg_dict).reset_index()


    ## Add the new columns we need for building / calculating matches for our dataframes
    new_df[time_diff_col] = [{} for _ in range(len(new_df))]
    new_df[match_group_col] = 0

    ## If we have rows that equal the number of message we were looking for, then set them aside
    complete_df = new_df[new_df[groupby_cnt_col] == len(prefixes)]

    new_df = new_df[new_df[groupby_cnt_col] < len(prefixes)]
    print("\n")
    print(f"==================== partial df {len(new_df)} records ====================")
    display(new_df)

    ## ======== Calculate the time difference between matching rows ========
    ## Loop through all of the rows and calculate the different time differences between the
    ##     rows that were grouped together
    for index, row in new_df.iterrows():
        #print(f"index: {index}")
        time_differnces = {}
        for index2, row2 in new_df[index + 1:].iterrows():
            #print(f"index2: {index2}")
            if index != index2:
                correct_group = True
                for col in groupby:
                    #print(f"{index}.{row[col]} != {index2}.{row2[col]} ({row[col] != row2[col]})")
                    if row2[col] != row[col]:
                        correct_group = False
                        break

                if correct_group:
                    time_differnces[index2] = (row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000
                    new_df.at[index, match_group_col] += 1
                    new_df.at[index, time_diff_col][index2] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)
                    new_df.at[index2, match_group_col] += 1
                    new_df.at[index2, time_diff_col][index] = abs((row2[timestmp_col] - row[timestmp_col]).total_seconds() * 1000)

    ## Take any records that had no matches and move them to the complete dataframe
    tmp_df = new_df[new_df[match_group_col] == 0]
    complete_df = pd.concat([complete_df, tmp_df])

    ## Drop the records where there was no match found
    new_df = new_df[new_df[match_group_col] > 0]

    print("\n")
    print(f"==================== unmatched/complete df {len(complete_df)} records, with timedifferences ====================")
    display(complete_df)

    print("\n")
    print(f"==================== partial df {len(new_df)} records, with timedifferences ====================")
    display(new_df)


    ## ======== Loop through all of our unmatched records and see about ========
    ## Setup our merged dataframe
    merged = {}
    for column in new_df.columns:
        merged[column] = []
    rows_inspected = []
    for index, row in new_df.iterrows():
        ## If we've already used this row, then move along
        if index not in rows_inspected:
            ## Matching Row
            time_differnces = row[time_diff_col]
            min_time_diff = min(time_differnces.values())
            key = [key for key, value in time_differnces.items() if value == min_time_diff][0]
            match_row = new_df.loc[key]

            #rows_inspected.append(index)

    return None

prefixes = ["1. ", "2. ", "3. ", "4. ", "5. "]
new_columns = ["msg_1", "msg_2", "msg_3", "msg_4", "msg_5"]
convert_messages_to_ten_line(
    df, prefixes=prefixes, new_columns=new_columns, 
    groupby=["room", "username"], timestmp_col="timestamp", msg_col="message"
)



==================== partial df 470 records ====================


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5,msg_cnt,time_diff,match_group
0,Room1,UserA,2026-03-30 14:10:03.332442,Tango8,NaN,NaN,NaN,NaN,1,{},0
1,Room1,UserA,2026-03-30 14:10:03.344442,NaN,TEST3,NaN,NaN,NaN,1,{},0
2,Room1,UserA,2026-03-30 14:10:03.350442,NaN,NaN,TEST9,NaN,NaN,1,{},0
3,Room1,UserA,2026-03-30 14:10:03.356442,NaN,NaN,NaN,None4,NaN,1,{},0
4,Room1,UserA,2026-03-30 14:10:03.360442,NaN,NaN,NaN,NaN,Time to Completion9: 2026-03-30T14:12:57.360442,1,{},0
...,...,...,...,...,...,...,...,...,...,...,...
465,Room5,UserE,2026-03-30 18:40:13.287442,Delta7,NaN,NaN,NaN,NaN,1,{},0
466,Room5,UserE,2026-03-30 18:40:13.299442,NaN,N/A9,NaN,NaN,NaN,1,{},0
467,Room5,UserE,2026-03-30 18:40:13.314442,NaN,NaN,TEST8,NaN,NaN,1,{},0
468,Room5,UserE,2026-03-30 18:40:13.322442,NaN,NaN,NaN,N/A9,NaN,1,{},0




==================== unmatched/complete df 0 records, with timedifferences ====================


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5,msg_cnt,time_diff,match_group




==================== partial df 470 records, with timedifferences ====================


,room,username,timestamp,msg_1,msg_2,msg_3,msg_4,msg_5,msg_cnt,time_diff,match_group
0,Room1,UserA,2026-03-30 14:10:03.332442,Tango8,NaN,NaN,NaN,NaN,1,"{1: 12.0, 2: 18.0, 3: 24.0, 4: 28.0, 5: 164020...",14
1,Room1,UserA,2026-03-30 14:10:03.344442,NaN,TEST3,NaN,NaN,NaN,1,"{0: 12.0, 2: 6.0, 3: 12.0, 4: 16.0, 5: 1640193...",14
2,Room1,UserA,2026-03-30 14:10:03.350442,NaN,NaN,TEST9,NaN,NaN,1,"{0: 18.0, 1: 6.0, 3: 6.0, 4: 10.0, 5: 1640187....",14
3,Room1,UserA,2026-03-30 14:10:03.356442,NaN,NaN,NaN,None4,NaN,1,"{0: 24.0, 1: 12.0, 2: 6.0, 4: 4.0, 5: 1640181....",14
4,Room1,UserA,2026-03-30 14:10:03.360442,NaN,NaN,NaN,NaN,Time to Completion9: 2026-03-30T14:12:57.360442,1,"{0: 28.0, 1: 16.0, 2: 10.0, 3: 4.0, 5: 1640177...",14
...,...,...,...,...,...,...,...,...,...,...,...
465,Room5,UserE,2026-03-30 18:40:13.287442,Delta7,NaN,NaN,NaN,NaN,1,"{446: 21453575.0, 447: 21453570.0, 448: 214535...",23
466,Room5,UserE,2026-03-30 18:40:13.299442,NaN,N/A9,NaN,NaN,NaN,1,"{446: 21453587.0, 447: 21453582.0, 448: 214535...",23
467,Room5,UserE,2026-03-30 18:40:13.314442,NaN,NaN,TEST8,NaN,NaN,1,"{446: 21453602.0, 447: 21453597.0, 448: 214535...",23
468,Room5,UserE,2026-03-30 18:40:13.322442,NaN,NaN,NaN,N/A9,NaN,1,"{446: 21453610.0, 447: 21453605.0, 448: 214535...",23
